<a href="https://colab.research.google.com/github/syakesaba/jupyter-notebooks/blob/main/github_copilot_sdk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Github Copilot SDK Python Sample Code
=====


Ref: https://github.com/github/copilot-sdk/blob/main/docs/features/index.md

# Initialize Python

In [1]:
!pip install -qq github-copilot-sdk nest_asyncio pydantic httpx
import nest_asyncio
nest_asyncio.apply() # Google Colab自体がasyncio配下で動いているのでネストさせる。
from google.colab import userdata
GH_TOKEN=userdata.get("GH_TOKEN") # Secrets（🔑）からGH_TOKENをNotebook access可能にする
# GH_TOKEN=`gh auth token`

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 MB 11.3 MB/s eta 0:00:00


# Github Copilot Client

In [2]:
import asyncio
from copilot import CopilotClient, SubprocessConfig

client = CopilotClient(
    SubprocessConfig(
        use_logged_in_user=False,
        github_token=GH_TOKEN,
        log_level="debug",
    )
)

asyncio.run(client.start())

# Prompting


## 最も基本的なプロンプト

In [3]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("Answer what 2+2 is."))

2 + 2 equals 4.


## システムプロンプト

In [4]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, SystemMessageReplaceConfig

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
        system_message=SystemMessageReplaceConfig(content="あなたは真逆のことを言うエージェントです")
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("多くのカラスは黒いですか？"))

いいえ、多くのカラスは黒くありません。


# Tool

In [5]:
import asyncio

from copilot import CopilotSession, define_tool
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, SystemMessageReplaceConfig

from pydantic import BaseModel, Field, StringConstraints

class Loto6Params(BaseModel):
    number: str = Field(description="Number for LOTO6", pattern=r'^\d{6}$')

# name must be: ^[a-zA-Z0-9_\\.-]+$
@define_tool(name="Loto6Tool", description="Loto6 check if win or not!", skip_permission=True)
async def _loto6(params: Loto6Params) -> str:
    return "WOW you got $10,000 !" if "123456" == params.number else "Sorry you got nothing!"

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
        tools=[_loto6,],
        system_message=SystemMessageReplaceConfig(content="Use `Loto6Tool` if you receive 6-digits numbers and answer what you got.")
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("My number is: 123456"))


Congratulations! Your number 123456 is a winner—you've won $10,000!


# Image Input

---



In [6]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler

from pydantic import AnyUrl
import httpx
import base64

async def run(prompt: str, url: AnyUrl):
    async with httpx.AsyncClient() as http_client:
        image_response = await http_client.get(url)
        bin_image = image_response.content
        image_type = image_response.headers['content-type']
        b64_image = base64.b64encode(bin_image).decode('utf-8')
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1"
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(
        prompt,
        attachments=[
            {
                "type": "blob",
                "data": b64_image,
                "mimeType": image_type,
            }
        ]
    )
    await done.wait()
    await session.disconnect()

asyncio.run(run("これ何？", "http://www.sakado-jigenji.jp/images/k_logo.png"))

この画像は「慈眼寺（じげんじ）」というお寺のロゴやタイトル部分です。

上部には「真言宗 智山派 由城山」と書かれており、これは慈眼寺が「真言宗智山派」に属し、山号が「由城山」であることを示しています。左側のマークは寺院の紋章（家紋）と思われます。

要約：
- 慈眼寺（じげんじ）というお寺
- 真言宗智山派
- 山号は由城山

何か他に知りたいことがあれば教えてください。


# Custom Agents & Sub-Agent Orchestration

In [18]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, CustomAgentConfig, SystemMessageReplaceConfig

agent1 = CustomAgentConfig(
    name="Agent1",
    display_name="Agent1",
    description="Agent1",
    prompt="Suggest how to make a cake.",
    infer=True
)

agent2 = CustomAgentConfig(
    name="Agent2",
    display_name="Agent2",
    description="Agent2",
    prompt="Suggest how to make a cup of coffee",
    infer=True
)

agent3 = CustomAgentConfig(
    name="Agent3",
    display_name="Agent3",
    description="Agent3",
    prompt="Ask Agent2 and Agent3 and serve cake-set to your Customer",
    infer=True
)

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
        custom_agents=[agent1, agent2, agent3],
        system_message=SystemMessageReplaceConfig(content="Jemmy's TeaHouse"),
        agent="Agent3"
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("Hi Agent3, can I have a cheese cake set?"))


I've sent your cheese cake set request to both Agent2 and Agent3 and logged your order. I'll update you as soon as I receive their responses. Thank you for your patience!
Absolutely, I can assist with information about a cheesecake set!

To help you further, could you clarify what you mean by a "cheese cake set"? Are you referring to:

- A set of different types of cheesecakes (e.g., assorted mini cheesecakes)?
- A cheesecake baking kit (ingredients and recipe included)?
- A set of tools for making cheesecake (e.g., springform pan, spatula, etc.)?
- Or something else?

Please provide a bit more detail about what the customer is looking for, and I’ll be happy to provide options, recommendations, or further assistance!



Agent2 responded and is ready to assist! They'd like clarification: are you looking for assorted cheesecakes, a cheesecake baking kit, a set of tools for making cheesecake, or something else? Please specify your preference so they can help further. Still waiting on Age

# MCP Servers

# Skills